# 08 - Revue bancaire experte: IFRS 9, Bale et stress testing

            Ce notebook complete les notebooks 01 a 07 avec une vue de plateforme bancaire:
            date d'arrete, segmentation reglementaire, RWA, capital stack, stress testing a 1 an,
            projection T1-T5, gouvernance modele et audit trail.

            Objectif de soutenance: montrer que le projet ne calcule pas seulement une ECL totale,
            mais simule une decision bancaire: perte attendue l'annee prochaine, resistance capital
            et actions de management.

## Resultats generes automatiquement

Cette synthese est inseree par `scripts/generate_banking_expert_review.py` pour que
les resultats soient visibles dans le notebook meme avant re-execution.

### Capital stack Bale

```text
  EAD_M$   RWA_M$  RWA_density_%  CET1_ratio_%  Total_capital_ratio_%  Leverage_ratio_%
34004.21 29063.66          85.47          12.0                   15.5             10.26
```

### Matrice stress testing a 1 an - scenarios les plus critiques

```text
              Scenario  Expected_loss_Y1_M$  CET1_post_stress_%  Capital_shortfall_7pct_M$                                                                                   Management_action
        Crise COVID-19              9267.44              -9.661                    5172.85 Recapitaliser, vendre actifs risques, suspendre origination risquee et activer plan de remediation.
            Crise 2008              8125.72              -7.475                    4261.05 Recapitaliser, vendre actifs risques, suspendre origination risquee et activer plan de remediation.
                Severe              5619.74              -1.555                    2455.35 Recapitaliser, vendre actifs risques, suspendre origination risquee et activer plan de remediation.
             Choc Taux              4958.18               0.060                    1956.66 Recapitaliser, vendre actifs risques, suspendre origination risquee et activer plan de remediation.
               Adverse              4700.91               0.699                    1774.96 Recapitaliser, vendre actifs risques, suspendre origination risquee et activer plan de remediation.
Inflation Ukraine 2022              2665.53               5.939                     289.72               Limiter dividendes, reduire croissance RWA, renforcer recouvrement et pricing risque.
      Choc Marche 2018               934.29              10.573                       0.00                                Maintenir coussins, suivi normal et revue trimestrielle des limites.
```

### Projection dynamique T1-T5 - crise 2008

```text
 year   ECL_M$   RWA_M$  CET1_%  PD_mean_%  UNRATE  GDPC1_growth
    1 11928.76 29437.31  -7.475     90.112  10.000       -4.3000
    2 13334.69 30336.66 -10.497     87.736   9.175       -3.4337
    3 13843.52 30948.96 -11.440     84.622   8.350       -2.5675
    4 13540.04 30946.31 -11.441     80.628   7.525       -1.7012
    5 12191.08 30383.87 -11.653     75.047   6.700       -0.8350
```

In [2]:
import sys

from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.banking_core import (
                capital_actions_from_shortfall,
                compute_basel_metrics,
                enrich_banking_view,
                ensure_model_registry,
                load_audit_log,
                log_audit_event,
                segment_summary,
            )
from src.scenario_projection import compute_dynamic_projection
from src.stress_scenarios import ACTIVE_STRESS_SCENARIOS, SCENARIO_LABELS

OUTPUT_DIR = ROOT / "outputs"
RESULTS_DIR = OUTPUT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)



## 1. Portefeuille a date d'arrete

            Une banque ne travaille pas sur tout l'historique brut. Elle fige un portefeuille a une
            date de reporting, puis calcule ECL, RWA et stress testing sur cette photographie.

In [4]:
df_full = pd.read_parquet(OUTPUT_DIR / "loan_clean.parquet")
ecl = pd.read_parquet(OUTPUT_DIR / "ecl_baseline.parquet")
cols = [c for c in ["issue_d", "purpose", "grade", "stage", "loan_amnt", "EAD", "LGD", "term_months"] if c in df_full.columns]
df = df_full[cols].copy()
df["PD_pred"] = ecl["PD_pred"].values
df["ECL_ifrs9"] = ecl["ECL_ifrs9"].values
del df_full, ecl

reporting_date = pd.to_datetime(df["issue_d"]).max() if "issue_d" in df.columns else pd.Timestamp.today()
banking_df, reporting_date = enrich_banking_view(df, reporting_date)

print(f"Date d'arrete: {reporting_date.date()}")
print(f"Nombre de prets: {len(banking_df):,}")
print(f"EAD total: {banking_df['EAD'].sum()/1e6:,.1f} M$")
print(f"ECL baseline: {banking_df['ECL_ifrs9'].sum()/1e6:,.1f} M$")
banking_df.head()

Date d'arrete: 2018-12-01
Nombre de prets: 2,260,668
EAD total: 34,004.2 M$
ECL baseline: 3,803.0 M$


,issue_d,grade,stage,loan_amnt,EAD,LGD,term_months,PD_pred,ECL_ifrs9,reporting_date,regulatory_segment,risk_weight,RWA
0,2018-12-01,C,1,2500,2500,0.612705,36.0,0.495695,253.095659,2018-12-01,retail_consumer,0.75,1875.0
1,2018-12-01,D,1,30000,30000,0.649445,60.0,0.627705,2445.960797,2018-12-01,retail_consumer,0.75,22500.0
2,2018-12-01,D,1,5000,5000,0.649445,36.0,0.583070,631.120422,2018-12-01,retail_consumer,0.75,3750.0
3,2018-12-01,D,1,4000,4000,0.649445,36.0,0.584811,506.404018,2018-12-01,retail_consumer,0.75,3000.0
4,2018-12-01,C,1,30000,30000,0.612705,60.0,0.578466,2126.574071,2018-12-01,retail_consumer,0.75,22500.0


## 2. Segmentation reglementaire et RWA

            La segmentation permet de sortir du simple portefeuille agrege. Elle porte la logique
            Bale: chaque segment a un risk weight et contribue differemment au RWA.

In [5]:
seg = segment_summary(banking_df)
seg.to_csv(RESULTS_DIR / "regulatory_segment_summary.csv", index=False)
seg

,regulatory_segment,loans,EAD_M,RWA_M,PD_mean,LGD_mean,ECL_M,RWA_density_%
0,defaulted,262447,4073.8485,6110.7728,0.5503,0.6251,1506.0826,150.0
1,high_risk,35056,673.4946,1010.2419,0.7349,0.7303,88.0792,150.0
2,retail_consumer,1963165,29256.8655,21942.6491,0.4347,0.5866,2208.8761,75.0


In [6]:
basel_metrics = compute_basel_metrics(banking_df, cet1_ratio=0.12)
basel_df = pd.DataFrame([basel_metrics])
basel_df.to_csv(RESULTS_DIR / "basel_capital_metrics.csv", index=False)
basel_df.T

,0
EAD_M$,34004.21
RWA_M$,29063.66
RWA_density_%,85.47
CET1_capital_M$,3487.64
AT1_capital_M$,435.95
Tier2_capital_M$,581.27
CET1_ratio_%,12.00
Tier1_ratio_%,13.50
Total_capital_ratio_%,15.50
Leverage_ratio_%,10.26


## 3. Stress testing a 1 an: perte, CET1 et decision

            Le coeur bancaire du stress testing est la question suivante:
            si un choc arrive l'annee prochaine, combien la banque doit provisionner,
            est-ce que le CET1 resiste, et quelle action doit etre declenchee?

In [7]:
rows = []
for scenario_key, scenario in ACTIVE_STRESS_SCENARIOS.items():
                projection, stage_projection = compute_dynamic_projection(
                    banking_df,
                    scenario,
                    horizon_years=1,
                    profile="front_loaded",
                    cet1_initial=0.12,
                    tax_rate=0.30,
                    scenario_name=scenario_key,
                )
                row = projection.iloc[0].to_dict()
                shortfall = max((7.0 - row["CET1_%"]) / 100 * row["RWA_M$"], 0)
                rows.append({
                    "scenario_key": scenario_key,
                    "Scenario": SCENARIO_LABELS.get(scenario_key, scenario_key),
                    "Expected_loss_Y1_M$": row["Annual_provision_M$"],
                    "ECL_Y1_M$": row["ECL_M$"],
                    "RWA_Y1_M$": row["RWA_M$"],
                    "CET1_post_stress_%": row["CET1_%"],
                    "PD_mean_%": row["PD_mean_%"],
                    "LGD_mean_%": row["LGD_mean_%"],
                    "Stage3_%": stage_projection.iloc[0]["Stage 3_%"],
                    "Capital_shortfall_7pct_M$": round(shortfall, 2),
                    "Management_action": capital_actions_from_shortfall(row["CET1_%"], shortfall),
                })

stress_matrix = pd.DataFrame(rows).sort_values("CET1_post_stress_%")
stress_matrix.to_csv(RESULTS_DIR / "one_year_stress_decision_matrix.csv", index=False)
stress_matrix

,scenario_key,Scenario,Expected_loss_Y1_M$,ECL_Y1_M$,RWA_Y1_M$,CET1_post_stress_%,PD_mean_%,LGD_mean_%,Stage3_%,Capital_shortfall_7pct_M$,Management_action
6,covid_2020,Crise COVID-19,9267.44,13070.47,31047.67,-9.661,93.404,71.822,17.01,5172.85,"Recapitaliser, vendre actifs risques, suspendr..."
5,crisis_2008,Crise 2008,8125.72,11928.76,29437.31,-7.475,90.112,73.340,15.92,4261.05,"Recapitaliser, vendre actifs risques, suspendr..."
1,severe,Severe,5619.74,9422.78,28700.72,-1.555,79.013,67.567,14.28,2455.35,"Recapitaliser, vendre actifs risques, suspendr..."
2,rate_shock,Choc Taux,4958.18,8761.22,28193.99,0.060,66.434,72.482,13.40,1956.66,"Recapitaliser, vendre actifs risques, suspendr..."
0,adverse,Adverse,4700.91,8503.94,28169.44,0.699,70.849,66.254,13.67,1774.96,"Recapitaliser, vendre actifs risques, suspendr..."
4,ukraine_2022,Inflation Ukraine 2022,2665.53,6468.57,27306.17,5.939,56.072,66.283,12.88,289.72,"Limiter dividendes, reduire croissance RWA, re..."
3,market_2018,Choc Marche 2018,934.29,4737.33,26799.52,10.573,47.230,60.394,12.44,0.00,"Maintenir coussins, suivi normal et revue trim..."


## 4. Projection dynamique T1-T5

            Cette section montre la trajectoire pluriannuelle d'un scenario severe: macro,
            ECL, RWA, CET1 et migration de stages. Elle repond a la critique du calcul
            uniquement total et non temporel.

In [8]:
projection, stage_projection = compute_dynamic_projection(
                banking_df,
                ACTIVE_STRESS_SCENARIOS["crisis_2008"],
                horizon_years=5,
                profile="front_loaded",
                cet1_initial=0.12,
                tax_rate=0.30,
                scenario_name="crisis_2008_expert",
            )
projection.to_csv(RESULTS_DIR / "dynamic_projection_expert.csv", index=False)
stage_projection.to_csv(RESULTS_DIR / "dynamic_stage_projection_expert.csv", index=False)
projection

,year,scenario,profile,severity_score,EAD_M$,RWA_M$,RWA_density_%,ECL_M$,Delta_ECL_vs_T0_M$,Annual_provision_M$,PD_mean_%,LGD_mean_%,CET1_%,Capital_M$,UNRATE,GDPC1_growth,FEDFUNDS,HY_SPREAD,CPI_change,DRCCLACBS
0,1,crisis_2008_expert,front_loaded,0.8979,33160.90,29437.31,88.77,11928.76,8125.72,8125.72,90.112,73.340,-7.475,-2200.36,10.000,-4.3000,0.2500,18.5000,-0.40,6.7700
1,2,crisis_2008_expert,front_loaded,0.7744,32086.82,30336.66,94.55,13334.69,9531.66,1405.94,87.736,71.448,-10.497,-3184.52,9.175,-3.4337,0.5594,16.4375,-0.07,6.1829
2,3,crisis_2008_expert,front_loaded,0.6510,30803.99,30948.96,100.47,13843.52,10040.48,508.82,84.622,69.543,-11.440,-3540.70,8.350,-2.5675,0.8688,14.3750,0.26,5.5957
3,4,crisis_2008_expert,front_loaded,0.5275,29338.65,30946.31,105.48,13540.04,9737.00,0.00,80.628,67.626,-11.441,-3540.70,7.525,-1.7012,1.1781,12.3125,0.59,5.0086
4,5,crisis_2008_expert,front_loaded,0.4041,27720.33,30383.87,109.61,12191.08,8388.04,0.00,75.047,65.698,-11.653,-3540.70,6.700,-0.8350,1.4875,10.2500,0.92,4.4215


In [9]:
stage_projection

,year,Stage 1_%,Stage 2_%,Stage 3_%,migrated_loans,scenario,severity_score,S1_to_S2_%,S1_to_S3_%,S2_to_S3_%,S3_maintained_%,n_migrated
0,1,69.76,14.32,15.92,409423,crisis_2008_expert_Y1,0.898,15.5,4.6,38.8,98.8,409423
1,2,57.27,19.03,23.70,417963,crisis_2008_expert_Y2,0.774,14.2,4.1,36.0,98.5,417963
2,3,48.24,20.05,31.71,378533,crisis_2008_expert_Y3,0.651,13.0,3.6,33.3,98.3,378533
3,4,41.66,19.61,38.73,328720,crisis_2008_expert_Y4,0.528,11.8,3.1,30.6,98.1,328720
4,5,36.87,18.70,44.44,282182,crisis_2008_expert_Y5,0.404,10.5,2.6,27.9,97.8,282182


## 5. Gouvernance: model registry et audit trail

            Une plateforme bancaire doit tracer la version des modeles, le statut de validation,
            le scenario execute et les sorties principales.

In [10]:
registry = ensure_model_registry()
registry

,model_id,version,owner,validation_status,approval_status,last_review_date
0,PD_CHAMPION,v1.0,Risk Modeling,Validated for PFA prototype,Model Committee simulated,2026-05-08
1,SATELLITE_MACRO_PD,v1.0,Stress Testing,Backtesting required before production,Conditional approval,2026-05-08
2,BASEL_RWA_STANDARDIZED,v1.0,Regulatory Capital,Rule-based benchmark,Approved for simulation,2026-05-08


In [11]:
worst = stress_matrix.iloc[0]
run_id = log_audit_event(
                event_type="notebook_08_banking_expert_review",
                user="notebook_user",
                parameters={
                    "reporting_date": str(reporting_date.date()),
                    "n_scenarios": len(stress_matrix),
                    "capital_framework": "Basel standardized simplified",
                },
                outputs={
                    "worst_scenario": worst["Scenario"],
                    "expected_loss_y1_m": worst["Expected_loss_Y1_M$"],
                    "cet1_post_stress_pct": worst["CET1_post_stress_%"],
                    "management_action": worst["Management_action"],
                },
            )
print(f"Run audite: {run_id}")
load_audit_log(limit=10)

Run audite: b2d3019a-3f6b-4afa-bebc-1bdca5e870fc


,run_id,timestamp_utc,event_type,user,parameters,outputs
0,8e6934a8-fa28-4d5b-9105-0c657ebd5c31,2026-05-08T09:24:04.403490+00:00,script_banking_expert_review,pipeline,"{'reporting_date': '2018-12-01', 'n_scenarios'...","{'worst_scenario': 'Crise COVID-19', 'expected..."
1,ef74c609-bb18-40f6-9269-5112e73343b5,2026-05-08T09:26:05.249147+00:00,script_banking_expert_review,pipeline,"{'reporting_date': '2018-12-01', 'n_scenarios'...","{'worst_scenario': 'Crise COVID-19', 'expected..."
2,84e56614-0a21-4b67-a53d-e1f31e4c5fda,2026-05-08T09:27:41.163586+00:00,script_banking_expert_review,pipeline,"{'reporting_date': '2018-12-01', 'n_scenarios'...","{'worst_scenario': 'Crise COVID-19', 'expected..."
3,a30f269e-25b9-4c9c-b599-d4e9f66ce770,2026-05-08T09:36:38.993613+00:00,script_banking_expert_review,pipeline,"{'reporting_date': '2018-12-01', 'n_scenarios'...","{'worst_scenario': 'Crise COVID-19', 'expected..."
4,0764bbd3-99a5-4454-bc25-8521348c7a45,2026-05-08T09:39:12.389378+00:00,script_banking_expert_review,pipeline,"{'reporting_date': '2018-12-01', 'n_scenarios'...","{'worst_scenario': 'Crise COVID-19', 'expected..."
5,974524fc-babc-47a5-9c90-4ddcc8203114,2026-05-08T10:20:17.442170+00:00,one_year_stress_matrix,streamlit_user,"{'scenario': 'Adverse', 'horizon_years': 1, 'c...","{'worst_scenario': 'Crise 2008', 'worst_loss_y..."
6,b2d3019a-3f6b-4afa-bebc-1bdca5e870fc,2026-05-08T11:51:59.198953+00:00,notebook_08_banking_expert_review,notebook_user,"{'reporting_date': '2018-12-01', 'n_scenarios'...","{'worst_scenario': 'Crise COVID-19', 'expected..."


## Conclusion bancaire

            Le projet couvre maintenant une chaine complete:

            macro scenario -> PD/LGD/EAD -> ECL -> RWA -> CET1 -> decision management -> audit trail.

            Les limites a expliquer restent: risk weights simplifiees, donnees Lending Club non bancaires
            au sens prudentiel complet, absence de vraie base comptable et calibration encore academique.